In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from datetime import datetime
import tempfile, os
import uuid
import json
import matplotlib.pyplot as plt
import os, re
import uuid
from matplotlib.figure import Figure

In [ ]:
# --- Configuración de persistencia ---
PERSIST_DIR = "./persist"
CSV_PATH = os.path.join(PERSIST_DIR, "state.csv")
JSON_PATH = os.path.join(PERSIST_DIR, "state.json")

def ensure_persist_paths():
    os.makedirs(PERSIST_DIR, exist_ok=True)
    return CSV_PATH, JSON_PATH

CSV_PATH, JSON_PATH = ensure_persist_paths()

In [ ]:
def persist_state(state, csv_path=CSV_PATH, json_path=JSON_PATH):
    """Guarda el state (lista de dicts) a CSV y JSON."""
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    # CSV
    df = pd.DataFrame(state or [])
    df.to_csv(csv_path, index=False)

    # JSON (útil para mantener tipos y estructura)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(state or [], f, ensure_ascii=False, indent=2)

    return csv_path, json_path

In [ ]:
def load_state(csv_path=CSV_PATH, json_path=JSON_PATH):
    """Carga state desde JSON si existe, si no desde CSV."""
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")

    return []

In [ ]:
def fig_to_png_file(fig):
    os.makedirs("tmp_plots", exist_ok=True)
    path = f"tmp_plots/{uuid.uuid4().hex}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path

def fullscreen(fig):
    if fig is None:
        raise gr.Error("Primero genera una gráfica.")
    return fig_to_png_file(fig)

In [ ]:
def A_closed_form(a: float, k: float, eps: float = 1e-12) -> float:
    if a <= eps:
        return np.inf
    # estable
    return (-1.0 / np.expm1(-a)) + (1.0 / np.expm1(-(a + k)))

def W0_total_infinity_mixed(k, lam, lam_r, b, eps):
    b = float(np.clip(b, 0.0, 1.0))
    A_l  = A_closed_form(lam,  k, eps=eps)
    A_lr = A_closed_form(lam_r, k, eps=eps)
    if not (np.isfinite(A_l) and np.isfinite(A_lr)):
        return np.inf
    return (1.0 - b) * A_l + b * A_lr

In [ ]:
def compute_ramp_up(df: pd.DataFrame):
    eps = 1e-12
    T_cap = 24          # t está capado a 24 en la UI
    fallback_T = 500    # horizonte para fallback discreto (si W falla)

    if df is None or df.empty:
        return pd.DataFrame(), "No hay datos."

    # --- Inputs (vector) ---
    potential = df["potential_gain_annual_eur"].to_numpy(np.float64)
    r = np.clip(df["regression_to_mean"].to_numpy(np.float64), 0.0, 1.0)
    s = np.clip(df["structural_erosion"].to_numpy(np.float64), 0.0, 1.0)

    k   = np.clip(df["ramp_up"].to_numpy(np.float64), 0.0, 0.3)
    lam = np.clip(df["lambda_monthly"].to_numpy(np.float64), 0.0, 0.3)

    if "lambda_residual_monthly" in df.columns:
        lam_r = np.clip(df["lambda_residual_monthly"].to_numpy(np.float64), 0.0, 0.3)
    else:
        lam_r = lam.copy()

    t = np.clip(df["user_replacement_rate"].to_numpy(np.float64), 0.0, float(T_cap))
    t_int = np.floor(t).astype(int)  # ya está capado por t

    implemented = df["implemented"].to_numpy(bool)
    p = np.clip(df["residual_limit_proportion"].to_numpy(np.float64), 0.0, 0.7)

    # --- Factores informativos (para columnas) ---
    decay_factor   = np.exp(-lam * t)
    ramp_up_factor = 1.0 - np.exp(-k * t)
    total_factor   = r * ramp_up_factor * decay_factor * s

    # --- Outputs ---
    n = len(df)
    impact_tmax = np.zeros(n, dtype=np.float64)
    impact_acum = np.zeros(n, dtype=np.float64)
    W_used      = np.zeros(n, dtype=np.float64)

    for i in range(n):
        # a = np.exp(-lam_r[i] * j)
        if (not implemented[i]) or (potential[i] <= 0.0):
            continue

        rs = r[i] * s[i]
        b = float(np.clip(p[i] / rs, 0.0, 1.0)) if rs > eps else 0.0

        # 1) W0 infinito (sin rs)
        W0 = W0_total_infinity_mixed(k[i], lam[i], 0.3, b, eps)

        # 2) fallback discreto (sin rs)
        if (not np.isfinite(W0)) or (W0 <= eps):
            jj = np.arange(0, fallback_T + 1, dtype=np.float64)
            mix = b * 1 + (1.0 - b) * np.exp(-lam[i] * jj)
            w0j = (1.0 - np.exp(-k[i] * jj)) * mix
            W0 = float(np.sum(w0j))

        if (not np.isfinite(W0)) or (W0 <= eps):
            continue

        W_used[i] = W0  # (opcional) ahora esto es W0, renómbralo si quieres

        # 3) Foto en t (€/mes)
        ti = float(t[i])
        mix_t = b * 1 + (1.0 - b) * np.exp(-lam[i] * ti)
        w0_t = (1.0 - np.exp(-k[i] * ti)) * mix_t
        impact_tmax[i] = potential[i] * rs * (w0_t / W0)

        # 4) Acumulado hasta t_int
        j = np.arange(0, t_int[i] + 1, dtype=np.float64)
        mix_j = b *1  + (1.0 - b) * np.exp(-lam[i] * j)
        w0j = (1.0 - np.exp(-k[i] * j)) * mix_j
        impact_acum[i] = float(np.sum(potential[i] * rs * (w0j / W0)))

    out = df.copy()
    out["decay_factor_exp"]        = np.round(decay_factor, 2)
    out["ramp_up_factor_exp"]      = np.round(ramp_up_factor, 2)
    out["total_correction_factor"] = np.round(total_factor, 2)
    out["W_total_inf"]             = np.round(W_used, 6)
    out["impact_eur"]              = np.round(impact_tmax, 2)
    out["impacto_acumulado_eur"]   = np.round(impact_acum, 2)

    total_impact = float(np.sum(impact_acum))
    resumen = f"**Impacto total (acumulado)**: {total_impact:,.2f} €"
    resumen = resumen.replace(",", "X").replace(".", ",").replace("X", ".")

    return out, resumen


In [ ]:
def state_to_df(state):
  if not state:
      return pd.DataFrame(columns=[
          "initiative_id",
          "potential_gain_annual_eur",
          "regression_to_mean",
          "ramp_up",
          "lambda_monthly",
          "lambda_residual_monthly",
          "user_replacement_rate",
          "structural_erosion",
          "implemented",
          "residual_limit_proportion"
      ])
  return pd.DataFrame(state)

In [ ]:
# Etiquetas SOLO para la UI (no cambian las columnas internas)
UI_COLS = {
    "initiative_id": "ID del experimento",
    "potential_gain_annual_eur": "Potencial anual (€)",
    "regression_to_mean": "Regresión a la media",
    "ramp_up": "Adaptación del usuario",
    "lambda_monthly": "Decaimiento mensual",
    "lambda_residual_monthly": "Decaimiento residual",
    "user_replacement_rate": "Tasa de reposición usuarios",
    "structural_erosion": "Erosión estructural",
    "implemented": "Implementado",
    "residual_limit_proportion": "Proporción del límite residual",
    "impact_eur": "Impacto en t (€)",
    "impacto_acumulado_eur": "Impacto acumulado (€)",
    "W_total_inf": "W∞",
    "total_correction_factor": "Factor total",
    "decay_factor_exp": "Decaimiento",
    "ramp_up_factor_exp": "Ramp-up",
}

def df_for_ui(df):
    if df is None or df.empty:
        return df
    return df.rename(columns=UI_COLS)

In [ ]:
def upsert(state, initiative_id, potential, r, k, lambda_m, t, s, implemented, p):
    try:
        state = state or []
        initiative_id = (initiative_id or "").strip()

        # helper: recalcular y devolver SIEMPRE la tabla formateada
        def _ret(msg):
            df, resumen = compute_ramp_up(state_to_df(state))
            return state, f"{msg}\n\n{resumen}", df_for_ui(df), None

        if not initiative_id:
            return _ret("❌ Falta ID ❌")

        row = {
            "initiative_id": initiative_id,
            "potential_gain_annual_eur": float(potential or 0.0),
            "regression_to_mean": float(r or 0.0),
            "ramp_up": float(k or 0.0),
            "lambda_monthly": float(lambda_m or 0.0),
            "lambda_residual_monthly": float(0.3 or 0.0),
            "user_replacement_rate": float(t or 0.0),
            "structural_erosion": float(s or 0.0),
            "implemented": bool(implemented),
            "residual_limit_proportion": float(p or 0.0),
        }

        updated = False
        for i, r0 in enumerate(state):
            if r0.get("initiative_id") == initiative_id:
                state[i] = row
                updated = True
                break
        if not updated:
            state.append(row)

        # persist puede fallar (IO). Mejor que no tumbe la app:
        try:
            persist_state(state)
        except Exception as e:
            return _ret(f"⚠️ Guardado falló: {type(e).__name__}: {e}")

        df, resumen = compute_ramp_up(state_to_df(state))
        msg = f"✅ {'Actualizado' if updated else 'Añadido'} `{initiative_id}` ✅"
        return state, f"{msg}\n\n{resumen}", df_for_ui(df), None

    except Exception as e:
        return state or [], f"❌ Error: {type(e).__name__}: {e}", df_for_ui(pd.DataFrame()), None


In [ ]:
def export_csv_single(exp):
    if not exp or not exp.get("initiative_id"):
        return None
    df_in = pd.DataFrame([exp])
    df_out, _ = compute_ramp_up(df_in)
    path = "resultado_experimento.csv"
    df_out.to_csv(path, index=False)
    return path

In [ ]:
def plot_impact_and_cum_from_exp_figs(exp, horizon=None, fallback_T=500, eps=1e-12, T_cap=24):
    potential = float(exp["potential_gain_annual_eur"])
    r = np.clip(float(exp["regression_to_mean"]), 0.0, 1.0)
    s = np.clip(float(exp["structural_erosion"]), 0.0, 1.0)
    k = np.clip(float(exp["ramp_up"]), 0.0, 0.3)
    lam = np.clip(float(exp["lambda_monthly"]), 0.0, 0.3)
    lam_r = np.clip(float(exp.get("lambda_residual_monthly", lam)), 0.0, 0.3)

    t_user = float(exp["user_replacement_rate"])
    implemented = float(bool(exp.get("implemented", True)))
    p = np.clip(float(exp["residual_limit_proportion"]), 0.0, 0.7)

    t_int = int(np.clip(np.floor(t_user), 0, T_cap))

    if horizon is None:
        horizon = T_cap
    horizon = int(np.clip(horizon, 0, T_cap))
    t = np.arange(0, horizon + 1, dtype=np.float64)

    # --- salida por defecto ---
    impacto = np.zeros_like(t)
    impact_acum_curve = np.zeros_like(t)
    rs = r * s
    b = 0.0
    W0 = 0.0
    acum_compute = 0.0

    if implemented == 0.0 or potential <= 0.0 or (not np.isfinite(rs)) or rs <= eps:
        # todo cero
        pass
    else:
        # b = p/(rs) como en compute
        b = float(np.clip(p / rs, 0.0, 1.0))

        # W0 infinito (sin rs) EXACTO como compute
        # usa tus funciones reales (las mismas que compute_ramp_up)
        W0 = W0_total_infinity_mixed(k, lam, 0.3, b, eps=eps)

        # fallback discreto (sin rs) EXACTO como compute
        if (not np.isfinite(W0)) or (W0 <= eps):
            jj = np.arange(0, fallback_T + 1, dtype=np.float64)
            mix = b * 1 + (1.0 - b) * np.exp(-lam * jj)
            w0j = (1.0 - np.exp(-k * jj)) * mix
            W0 = float(np.sum(w0j))

        if np.isfinite(W0) and W0 > eps:
            # impacto por mes (euros): potential * rs * (w0/W0)
            mix_t = b * 1 + (1.0 - b) * np.exp(-lam * t)
            w0_t = (1.0 - np.exp(-k * t)) * mix_t
            impacto = potential * rs * (w0_t / W0)
            impact_acum_curve = np.cumsum(impacto)

            # acumulado EXACTO como tabla (0..t_int)
            j_ref = np.arange(0, t_int + 1, dtype=np.float64)
            mix_j = b * 1 + (1.0 - b) * np.exp(-lam * j_ref)
            w0j_ref = (1.0 - np.exp(-k * j_ref)) * mix_j
            acum_compute = float(np.sum(potential * rs * (w0j_ref / W0)))

    exp_label = exp.get("initiative_id", "(sin id)")

    # ---- FIG 1 ----
    fig1, ax1 = plt.subplots(figsize=(9, 4))
    ax1.plot(t, impacto, marker="o", label="I(t)", color="b")
    ax1.axhline((potential*p*r*s)/W0, linestyle="-", label="0",color="g")
    ax1.set_title(f"I(t) — {exp_label} (W0={W0:.4g}, b={b:.3g}, rs={rs:.3g}, t_int={t_int})")
    ax1.set_xlabel("t")
    ax1.set_ylabel("I(t) (€)")
    ax1.grid(True)
    ax1.legend()
    fig1.tight_layout()

    # ---- FIG 2 ----
    fig2, ax2 = plt.subplots(figsize=(9, 4))
    ax2.plot(t, impact_acum_curve, marker="o", label="Acumulado (∑ I hasta t)", color="r")

    if W0 > 0:
        ax2.scatter([t_int], [acum_compute], label=f"Tabla (0..t_int) = {acum_compute:,.2f} €")
        ax2.axhline(acum_compute, linestyle="--")

    ax2.axhline(0.0, linestyle=":", label="0")
    ax2.set_title(f"Impacto acumulado — {exp_label}")
    ax2.set_xlabel("t")
    ax2.set_ylabel("Impacto acumulado (€)")
    ax2.grid(True)
    ax2.legend()
    fig2.tight_layout()

    return fig1, fig2

def plot_by_id(exp_id: str):
    exp_id = (exp_id or "").strip()
    if not exp_id:
        raise gr.Error("Introduce un ID.")

    state = load_state()  # tu función existente
    exp = next((row for row in state if row.get("initiative_id") == exp_id), None)
    if exp is None:
        ids = [r.get("initiative_id") for r in state if r.get("initiative_id")]
        raise gr.Error(f"No existe el ID '{exp_id}'. Ejemplos: {ids[:10]}")

    fig1, fig2 = plot_impact_and_cum_from_exp_figs(exp)
    return fig1, fig2

def plot_by_id_with_state(exp_id):
    fig1, fig2 = plot_by_id(exp_id)
    return fig1, fig2, fig1, fig2

In [ ]:
def build_exp(initiative_id, potential, r, k, lambda_m, t, s, implemented, p):
    initiative_id = (initiative_id or "").strip()

    exp = {
        "initiative_id": initiative_id,
        "potential_gain_annual_eur": float(potential or 0.0),
        "regression_to_mean": float(r or 0.0),
        "ramp_up": float(k or 0.0),
        "lambda_monthly": float(lambda_m or 0.0),
        "lambda_residual_monthly": float(0.3),   # si quieres fijo como ahora
        "user_replacement_rate": float(t or 0.0),
        "structural_erosion": float(s or 0.0),
        "implemented": bool(implemented),
        "residual_limit_proportion": float(p or 0.0),
    }
    return exp


def recompute_single(exp):
    # tabla usando tu compute_ramp_up (con 1 fila)
    df_in = pd.DataFrame([exp]) if exp and exp.get("initiative_id") else pd.DataFrame()
    df_out, resumen = compute_ramp_up(df_in)
    # df_for_ui si lo estás usando para etiquetas/orden
    table_df = df_for_ui(df_out) if not df_out.empty else df_for_ui(pd.DataFrame())

    # gráficas
    fig1 = fig2 = None
    if exp and exp.get("initiative_id"):
        fig1, fig2 = plot_impact_and_cum_from_exp_figs(exp)

    return resumen, table_df, fig1, fig2


In [ ]:
def set_experiment(exp_state, initiative_id, potential, r, k, lambda_m, t, s, implemented, p):
    exp = build_exp(initiative_id, potential, r, k, lambda_m, t, s, implemented, p)
    resumen, table_df, fig1, fig2 = recompute_single(exp)

    return exp, resumen, table_df, fig1, fig2, fig1, fig2

In [ ]:
descriptions = {
    "r": """**r (Regresión a la media):**

Este parámetro corrige el optimismo inherente a las estimaciones.

Introduce una corrección sistemática al sesgo de sobreestimación, ajustando el impacto teórico hacia valores históricamente observables.

Modela la regresión a la media propia de iniciativas sin evidencia suficiente, reflejando que, en ausencia de señales fuertes, los resultados tienden a acercarse al rendimiento promedio.

No se recomoenda que este valor tome los valores 0, 1 ya que estos corresponden con escenarios tóricos pero no prácticos.
""",

    "k": """**k (Adaptación del usuario):**

Este parámetro cuantifica la velocidad de adaptación del usuario.

Un valor elevado indica una rápida adaptación de los usuarios a los cambios y mejoras del experimento, lo que se traduce en un mayor impacto observable en las etapas iniciales del mismo.

Valores más bajos indican mayor dificultad de los usuarios para adoptar las novedades desarrolladas por el experimento y, por tanto, el impacto tardará más tiempo en ser apreciado.
""",

    "lambda": """**λ (Decaimiento mensual):**

Este parámetro modela la velocidad con la que el impacto generado por la iniciativa se erosiona a lo largo del tiempo.

Refleja cómo los efectos iniciales tienden a perder fuerza a medida que los usuarios se familiarizan con el cambio o el sistema se satura.

Captura explícitamente fenómenos como la pérdida de *novelty*, la habituación de los usuarios y la reducción progresiva del diferencial frente al estado anterior.

Valores elevados indican que el impacto se deteriora con mayor rapidez; valores más bajos implican una pérdida de impacto más gradual.

Valores más bajos implican una pérdida de impacto más gradual, reflejando iniciativas cuyo efecto se mantiene durante más tiempo y se integra de forma más estable en el comportamiento de los usuarios.
""",

    "s": """**s (Erosión estructural):**

Este parámetro actúa como un multiplicador global que ajusta el impacto estimado para tener en cuenta fricciones no temporales inherentes al sistema.

Por ejemplo: dependencias técnicas, limitaciones operativas o factores externos que impiden materializar completamente el potencial teórico.

No se recomoenda que este valor tome los valores 0, 1 ya que estos corresponden con escenarios tóricos pero no prácticos.
""",

    "implemented": """**Implementado:**

Variable booleana: toma el valor 0 cuando el experimento no se está llevando a cabo y 1 cuando está implementado.
""",

    "potential": """**Potential Gain € (anual):**

Este parámetro implementa un máximo teórico al que el impacto puede aspirar.

No se debe interpretar como una promesa, sino como un techo.

El impacto está acotado superiormente por este parámetro.
""",

    "residual_limit_proportion": """**p (Proporción del límite residual):**

Naturaleza del Experimento: Novedad / Atención (Banners, Pop-ups, Promociones flash)

Valor Sugerido de p: 0 - 0.01

Razón Estratégica: El impacto depende del factor sorpresa; una vez el usuario se habitúa, el valor desaparece casi por completo.

Naturaleza del Experimento: Optimización de UX/CRO (Cambio de CTAs, orden de campos, copy de beneficios)
Valor Sugerido de p: 0.02 - 0.04


Razón Estratégica: Se elimina una fricción puntual. El impacto máximo decae, pero el sistema es intrínsecamente mejor que antes.

Naturaleza del Experimento: Cambio Estructural/Producto (Mejora de algoritmo, nueva funcionalidad core, propuesta de valor)

Valor Sugerido de p: 0.05 - 0.07
""",
    "user_replacement_rate": """**t (ciclos de reposición de usuarios):**

 Esta es la variable principal del modelo.

 Indica a cúantos cíclos de usuarios se somete el experimento.
"""
}


In [ ]:
def toggle_help(key, current):
    if current == key:
        return gr.update(value="", visible=False), None
    return gr.update(value=descriptions[key], visible=True), key

with gr.Blocks(title="Calculadora Impacto") as app:
  gr.HTML("""
    <style>
    .info-btn {
        width: 28px !important;
        min-width: 28px !important;
        max-width: 28px !important;
    }
    </style>
    """)
  gr.Markdown("# Calculadora Impacto \nModelo: `1/W × r × (1 - exp(-k·t)) × (b+(1-b)·exp(-λ·t)) × s`")

  exp_state = gr.State(None)
  current_help = gr.State(None)
  fig_state_1 = gr.State(None)
  fig_state_2 = gr.State(None)

  with gr.Row():
    with gr.Column(scale=1):

      initiative_id = gr.Textbox(label="ID del experimento", placeholder="PH-client-xxxx")
      with gr.Row():
        potential = gr.Number(label="Potential Gain € (anual)", value = 0)
        btn_potential = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        r = gr.Slider(0, 1, step=0.05, value=0.7, label="Regresión a la media (r)")
        btn_r = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        k = gr.Slider(0, 0.3, step=0.01, value=0.2, label="Adaptación del usuario (k)")
        btn_k = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        lambda_m = gr.Slider(0, 0.3, step=0.01, value=0.2, label="Decaimiento mensual λ")
        btn_lambda = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        t = gr.Slider(0, 24, step=1, label="Tasa reposición usuarios (t)", value=24)
        btn_t = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        s = gr.Slider(0, 1, step=0.05, value=0.95, label="Erosión estructural (s)")
        btn_s = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        implemented = gr.Checkbox(label="Implementado", value=True)
        btn_implemented = gr.Button("ℹ️", elem_classes="info-btn")
      with gr.Row():
        p = gr.Slider(0, 0.07, step=0.005, value=0, label="Proporción del límite residual (p)")
        btn_p = gr.Button("ℹ️", elem_classes="info-btn")

      btn_apply = gr.Button("Añadir / Actualizar experimento", variant="primary")

      with gr.Row():
          btn_big1 = gr.Button("Descargar I(t) a tamaño completo")
          btn_big2 = gr.Button("Descargar acumulado a tamaño completo")
      big_file = gr.File(label="Abrir en otra pestaña (pantalla completa)")

      with gr.Row():
          btn_export = gr.Button("Exportar CSV", variant="secondary")
      csv_out = gr.File(label="Descargar CSV")

    with gr.Column(scale=3):
      resumen = gr.Markdown()
      table = gr.DataFrame(label="Resultados", interactive=False)
      out1 = gr.Plot(label="I(t)")
      out2 = gr.Plot(label="Impacto acumulado")
      help_panel = gr.Markdown("", visible=False)

    btn_apply.click(
        fn=set_experiment,
        inputs=[exp_state, initiative_id, potential, r, k, lambda_m, t, s, implemented, p],
        outputs=[exp_state, resumen, table, out1, out2, fig_state_1, fig_state_2],
    )

    btn_export.click(
        export_csv_single,
        inputs = [exp_state],
        outputs = [csv_out]
    )

    btn_r.click(
        lambda current: toggle_help("r", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_potential.click(
        lambda current: toggle_help("potential", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_k.click(
        lambda current: toggle_help("k", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_lambda.click(
        lambda current: toggle_help("lambda", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_s.click(
        lambda current: toggle_help("s", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_implemented.click(
        lambda current: toggle_help("implemented", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_p.click(
        lambda current: toggle_help("residual_limit_proportion", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )

    btn_t.click(
        lambda current: toggle_help("user_replacement_rate", current),
        inputs=[current_help],
        outputs=[help_panel, current_help],
    )


    # --- Botones fullscreen/descarga ---
    btn_big1.click(fullscreen, inputs=fig_state_1, outputs=big_file)
    btn_big2.click(fullscreen, inputs=fig_state_2, outputs=big_file)

if __name__ == "__main__":
    app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30f24ef5461a52f69a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
